In [1]:
# =========================
# SETUP
# =========================

from collections import Counter
import re

# OPCIONAL: usar LLM pequeño (flan-t5 por defecto)
USE_LLM = True

if USE_LLM:
    try:
        from transformers import pipeline
        llm = pipeline("text2text-generation", model="google/flan-t5-small")
    except Exception as e:
        print("No se pudo cargar el modelo, fallback a heurísticas:", e)
        USE_LLM = False


# =========================
# HELPERS
# =========================

def is_empty(value):
    return value is None or str(value).strip() == ""


def majority_vote(labels):
    if not labels:
        return None
    return Counter(labels).most_common(1)[0][0]


def qwen_matches_majority(qwen_label, labels):
    mapping = {"NO": 0, "YES": 1}
    majority = majority_vote(labels)
    return mapping.get(qwen_label) == majority


# =========================
# HEURÍSTICA (fallback)
# =========================

def reasoning_has_content_heuristic(reasoning):
    if is_empty(reasoning):
        return False
    
    reasoning = reasoning.lower()
    
    if len(reasoning.split()) < 8:
        return False
    
    vague_phrases = [
        "no explicit",
        "not directly",
        "may need further analysis",
        "no clear indication",
        "not evident",
        "no indication",
        "not enough information"
    ]
    
    if any(p in reasoning for p in vague_phrases):
        return False
    
    return True


# =========================
# LLM CHECK
# =========================

def reasoning_has_content_llm(reasoning):
    prompt = f"""
Classify the following reasoning as:
- VALID: contains concrete, meaningful analysis
- VAGUE: generic, empty, or says nothing useful

Saying things like, I don't have enough information or I can't see anything on the image, is a wrong answer so you have to answer with VAGUE.

Reasoning:
\"\"\"{reasoning}\"\"\"

Answer only VALID or VAGUE.
"""
    try:
        output = llm(prompt, max_length=10)[0]["generated_text"].strip().upper()
        return "VALID" in output
    except:
        return reasoning_has_content_heuristic(reasoning)


def reasoning_has_content(reasoning):
    if USE_LLM:
        return reasoning_has_content_llm(reasoning)
    return reasoning_has_content_heuristic(reasoning)


# =========================
# MAIN VALIDATION
# =========================

def validate_sample(sample):
    result = {}
    
    # 1. Campos vacíos
    empty_fields = {
        "ocr": is_empty(sample.get("qwen_ocr")),
        "transcription": is_empty(sample.get("qwen_transcription")),
        "reasoning": is_empty(sample.get("qwen_reasoning")),
    }
    
    # 2. Label consistency
    label_match = qwen_matches_majority(
        sample.get("qwen_label"),
        sample.get("label", [])
    )
    
    # 3. Reasoning calidad
    reasoning_valid = reasoning_has_content(
        sample.get("qwen_reasoning", "")
    )
    
    # 4. Score global
    score = 0
    
    # penalizaciones
    score += sum(not v for v in empty_fields.values())  # +1 por campo OK
    score += int(label_match) * 2                      # más peso
    score += int(reasoning_valid) * 2
    
    result["empty_fields"] = empty_fields
    result["label_match"] = label_match
    result["reasoning_valid"] = reasoning_valid
    result["score"] = score
    
    return result


# =========================
# RUN SOBRE DATASET
# =========================

def validate_dataset(data):
    results = []
    
    for sample in data:
        res = validate_sample(sample)
        res["id"] = sample.get("id")
        results.append(res)
    
    return results


# =========================
# EJEMPLO DE USO
# =========================
import os
import json
with open(os.path.join("../data/processed", "test.json"), encoding="utf-8") as f:
    data = json.load(f)

results = validate_dataset(data)

# ver algunos resultados
# for r in results[:5]:
#     print(r)


# =========================
# FILTRADO ÚTIL
# =========================

def filter_bad_samples(results, min_score=6):
    return [r for r in results if r["score"] < min_score]


bad = filter_bad_samples(results)
print("Samples malos:", len(bad))

/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/turbotowerlnx/Documents/Master/ALC/ALC-Lab-Final/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Samples malos: 179


In [2]:
def filter_bad_samples(results, min_score=6):
    return [r for r in results if r["score"] < min_score]

bad = filter_bad_samples(results)
print("Samples malos:", len(bad))

print(bad)

Samples malos: 179
[{'empty_fields': {'ocr': False, 'transcription': False, 'reasoning': False}, 'label_match': False, 'reasoning_valid': True, 'score': 5, 'id': 120012}, {'empty_fields': {'ocr': False, 'transcription': False, 'reasoning': False}, 'label_match': False, 'reasoning_valid': True, 'score': 5, 'id': 120017}, {'empty_fields': {'ocr': False, 'transcription': False, 'reasoning': False}, 'label_match': False, 'reasoning_valid': True, 'score': 5, 'id': 120018}, {'empty_fields': {'ocr': False, 'transcription': False, 'reasoning': False}, 'label_match': False, 'reasoning_valid': True, 'score': 5, 'id': 120021}, {'empty_fields': {'ocr': False, 'transcription': False, 'reasoning': False}, 'label_match': False, 'reasoning_valid': True, 'score': 5, 'id': 120036}, {'empty_fields': {'ocr': False, 'transcription': False, 'reasoning': False}, 'label_match': False, 'reasoning_valid': True, 'score': 5, 'id': 120056}, {'empty_fields': {'ocr': False, 'transcription': False, 'reasoning': False

In [3]:
from collections import Counter
import numpy as np

# =========================
# STATS GENERALES
# =========================

def dataset_stats(results):
    stats = {}
    
    n = len(results)
    stats["total_samples"] = n
    
    # -------------------------
    # SCORE
    # -------------------------
    scores = [r["score"] for r in results]
    
    stats["score"] = {
        "mean": np.mean(scores),
        "median": np.median(scores),
        "min": np.min(scores),
        "max": np.max(scores),
    }
    
    # distribución de scores
    stats["score_distribution"] = dict(Counter(scores))
    
    # -------------------------
    # LABEL MATCH
    # -------------------------
    label_matches = [r["label_match"] for r in results]
    
    stats["label_match"] = {
        "correct": sum(label_matches),
        "incorrect": n - sum(label_matches),
        "accuracy": sum(label_matches) / n if n > 0 else 0
    }
    
    # -------------------------
    # REASONING
    # -------------------------
    reasoning_valid = [r["reasoning_valid"] for r in results]
    
    stats["reasoning"] = {
        "valid": sum(reasoning_valid),
        "invalid": n - sum(reasoning_valid),
        "valid_ratio": sum(reasoning_valid) / n if n > 0 else 0
    }
    
    # -------------------------
    # CAMPOS VACÍOS
    # -------------------------
    empty_ocr = sum(r["empty_fields"]["ocr"] for r in results)
    empty_transcription = sum(r["empty_fields"]["transcription"] for r in results)
    empty_reasoning = sum(r["empty_fields"]["reasoning"] for r in results)
    
    stats["empty_fields"] = {
        "ocr_empty": empty_ocr,
        "transcription_empty": empty_transcription,
        "reasoning_empty": empty_reasoning,
        "ocr_empty_ratio": empty_ocr / n if n > 0 else 0,
        "transcription_empty_ratio": empty_transcription / n if n > 0 else 0,
        "reasoning_empty_ratio": empty_reasoning / n if n > 0 else 0,
    }
    
    # -------------------------
    # CALIDAD GLOBAL
    # -------------------------
    good = sum(1 for r in results if r["score"] >= 6)
    ok = sum(1 for r in results if 4 <= r["score"] < 6)
    bad = sum(1 for r in results if r["score"] < 4)
    
    stats["quality"] = {
        "good": good,
        "ok": ok,
        "bad": bad,
        "good_ratio": good / n if n > 0 else 0,
        "bad_ratio": bad / n if n > 0 else 0,
    }
    
    return stats


# =========================
# PRINT BONITO
# =========================

def print_stats(stats):
    print("\n===== DATASET STATS =====\n")
    
    print(f"Total samples: {stats['total_samples']}")
    
    print("\n--- SCORE ---")
    for k, v in stats["score"].items():
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
    
    print("\nScore distribution:")
    for k, v in sorted(stats["score_distribution"].items()):
        print(f"  {k}: {v}")
    
    print("\n--- LABEL MATCH ---")
    print(f"Accuracy: {stats['label_match']['accuracy']:.3f}")
    print(f"Correct: {stats['label_match']['correct']}")
    print(f"Incorrect: {stats['label_match']['incorrect']}")
    
    print("\n--- REASONING ---")
    print(f"Valid ratio: {stats['reasoning']['valid_ratio']:.3f}")
    print(f"Valid: {stats['reasoning']['valid']}")
    print(f"Invalid: {stats['reasoning']['invalid']}")
    
    print("\n--- EMPTY FIELDS ---")
    ef = stats["empty_fields"]
    print(f"OCR empty: {ef['ocr_empty']} ({ef['ocr_empty_ratio']:.2%})")
    print(f"Transcription empty: {ef['transcription_empty']} ({ef['transcription_empty_ratio']:.2%})")
    print(f"Reasoning empty: {ef['reasoning_empty']} ({ef['reasoning_empty_ratio']:.2%})")
    
    print("\n--- QUALITY ---")
    q = stats["quality"]
    print(f"Good (6-7): {q['good']} ({q['good_ratio']:.2%})")
    print(f"OK (4-5): {q['ok']}")
    print(f"Bad (<4): {q['bad']} ({q['bad_ratio']:.2%})")


# =========================
# USO
# =========================

stats = dataset_stats(results)
print_stats(stats)


===== DATASET STATS =====

Total samples: 502

--- SCORE ---
mean: 6.281
median: 7.000
min: 2
max: 7

Score distribution:
  2: 1
  5: 178
  7: 323

--- LABEL MATCH ---
Accuracy: 0.645
Correct: 324
Incorrect: 178

--- REASONING ---
Valid ratio: 1.000
Valid: 502
Invalid: 0

--- EMPTY FIELDS ---
OCR empty: 2 (0.40%)
Transcription empty: 2 (0.40%)
Reasoning empty: 1 (0.20%)

--- QUALITY ---
Good (6-7): 323 (64.34%)
OK (4-5): 178
Bad (<4): 1 (0.20%)


In [5]:
# =========================
# CONFUSION YES/NO
# =========================

def confusion_stats(data, results):
    mapping = {"NO": 0, "YES": 1}
    
    confusion = {
        "TP": 0,  # YES correcto
        "TN": 0,  # NO correcto
        "FP": 0,  # predijo YES pero era NO
        "FN": 0   # predijo NO pero era YES
    }
    
    mismatch_list = []
    
    for sample, res in zip(data, results):
        qwen_label = mapping.get(sample.get("qwen_label"))
        majority = majority_vote(sample.get("label", []))
        
        if qwen_label is None or majority is None:
            continue
        
        # Casos
        if qwen_label == 1 and majority == 1:
            confusion["TP"] += 1
        elif qwen_label == 0 and majority == 0:
            confusion["TN"] += 1
        elif qwen_label == 1 and majority == 0:
            confusion["FP"] += 1
            mismatch_list.append({
                "id": sample.get("id"),
                "qwen_label": "YES",
                "majority": "NO"
            })
        elif qwen_label == 0 and majority == 1:
            confusion["FN"] += 1
            mismatch_list.append({
                "id": sample.get("id"),
                "qwen_label": "NO",
                "majority": "YES"
            })
    
    total = sum(confusion.values())
    
    stats = {
        "confusion": confusion,
        "total_evaluated": total,
        "accuracy": (confusion["TP"] + confusion["TN"]) / total if total > 0 else 0,
        "yes_as_no": confusion["FN"],  # eran YES pero dijo NO
        "no_as_yes": confusion["FP"],  # eran NO pero dijo YES
    }
    
    return stats, mismatch_list


# =========================
# PRINT BONITO
# =========================

def print_confusion(stats):
    c = stats["confusion"]
    
    print("\n===== CONFUSION MATRIX =====\n")
    
    print("                Majority")
    print("             NO       YES")
    print(f"QWEN NO   {c['TN']:6}   {c['FN']:6}")
    print(f"QWEN YES  {c['FP']:6}   {c['TP']:6}")
    
    print("\n--- METRICS ---")
    print(f"Accuracy: {stats['accuracy']:.3f}")
    print(f"NO → YES (FP): {stats['no_as_yes']}")
    print(f"YES → NO (FN): {stats['yes_as_no']}")
    print(f"Total evaluados: {stats['total_evaluated']}")


# =========================
# USO
# =========================

stats_conf, mismatches = confusion_stats(data, results)

print_confusion(stats_conf)

# =========================
# VER LISTA DE ERRORES
# =========================

def show_mismatches(mismatches, data, limit=10):
    print(f"\nMostrando {min(limit, len(mismatches))} errores:\n")
    
    id_to_sample = {s["id"]: s for s in data}
    
    for m in mismatches[:limit]:
        sample = id_to_sample.get(m["id"], {})
        
        print("ID:", m["id"])
        print("QWEN:", m["qwen_label"], "| Majority:", m["majority"])
        print("Transcription:", sample.get("qwen_transcription"))
        print("Reasoning:", sample.get("qwen_reasoning"), "...")
        print("-" * 50)


show_mismatches(mismatches, data, limit=10)


===== CONFUSION MATRIX =====

                Majority
             NO       YES
QWEN NO      193      109
QWEN YES      68      131

--- METRICS ---
Accuracy: 0.647
NO → YES (FP): 68
YES → NO (FN): 109
Total evaluados: 501

Mostrando 10 errores:

ID: 120012
QWEN: NO | Majority: YES
Transcription: Tú cuéntame, háblame claro, ¿cómo se llama la loca? A ver si le echo una acogida, ¿de dónde es, marico? Es la que ando enojada, marico. Tengo como un mes que no meto el loco, weón. Estoy que me mato, marico, estoy que enojada. Me cojo la gata esa que está en la casa, marico. No vale, mentira. No, marico, pero sí, weón. Quiero todo, todo, no, weón.
Reasoning: The visual frame shows a paused audio player with the text "*mi amigo el menos necesitado*" which translates to "*my friend the least needed*". The audio transcription contains colloquial language and references to sexual activity, but there are no explicit mentions of women being degraded, stereotyped, or objectified. The content seems 